# KC-Expansion audit — 

Reproduces all paper results from **raw** data:

1. **Shared interaction windows** — one canonical window catalogue is built from ordered question-level interaction IDs and reused by QL / EXP / LEGACY / SHUF. A shared row-budget guard may shorten a window for every condition together, but never splits an interaction. Thus paired conditions have exactly the same interaction coverage and target set.
2. **Exact all-in-one evaluation for all three models** — response embeddings are attached to their source rows. Attention is strictly causal, and all same-interaction key/value paths are excluded during all-in-one prediction; genuine past responses remain available to every sibling KC. DKT reads every sibling from the same state at the end of the previous interaction, without sibling updates. Executable tests check both target-label invariance and preceding-history sensitivity.
3. **Leak-free model selection** — every run early-stops on validation **all-in-one fused AUC** (identical to row AUC on QL data), so no model is selected via a leaky metric.
4. **5 seeds** (42, 123, 7, 2024, 31) with paired mean differences and 95% t-confidence intervals.
5. **Shortcut baseline** evaluated on each seed's test population (mean ± std reported) with cold-start behaviour documented in the code.

Required Kaggle inputs: raw `skill_builder_data.csv` (NOT `assist2009_aktf.csv` — broken ordering) and `assist2017.csv` / `anonymized_full_release_competition_dataset.csv`. Paths auto-detected.

In [2]:
# ================================================================
# 1. CONFIG — paths, hyperparameters, results store  (v3)
# ================================================================
import os, sys, json, math, glob, time, random, warnings, hashlib, platform
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def find_file(names):
    hits = []
    for n in names:
        hits += glob.glob(n)
        hits += glob.glob(f"/kaggle/input/**/{n}", recursive=True)
    hits = [h for h in hits if os.path.isfile(h)]
    return hits[0] if hits else None

RAW_A09 = find_file(["skill_builder_data.csv",
                     "skill_builder_data_corrected.csv"])
RAW_A17 = find_file(["anonymized_full_release_competition_dataset.csv",
                     "assist2017.csv"])
assert RAW_A09, ("ASSIST2009 raw file not found. Attach the Kaggle dataset with "
                 "skill_builder_data.csv (NOT assist2009_aktf.csv — broken ordering).")
assert RAW_A17, "ASSIST2017 raw file not found."
print("ASSIST2009 raw:", RAW_A09)
print("ASSIST2017 raw:", RAW_A17)

PIPELINE_VERSION = "v3.2-canonical-allinone-2026-07-22"
OUT_DIR = "paper_v3_out"
os.makedirs(OUT_DIR, exist_ok=True)
RESULTS_PATH = f"{OUT_DIR}/paper_v3_results.json"

RUN = dict(A=True, B=True, C=True, D=True, E=True, F=True, G=True)

SEEDS     = [42, 123, 7, 2024, 31]
MAIN_SEED = 42
MIN_SEQ   = 3

CFG = dict(
    d_model=128, n_heads=8, n_layers=2, dropout=0.1,
    # Canonical windows are specified in original interaction IDs and shared.
    max_inter=200, train_stride_inter=100, eval_stride_inter=200,
    max_rows=384,               # shared budget; shortens every condition together
    epochs=60, warmup=3, early_stop=10,
    lr=1e-3, wd=1e-4, label_smooth=0.05,
    ema_decay=0.995, clip=1.0, use_amp=True,
    n_att_bins=16,
)

def split_students(students, seed):
    """80/10/10 student split; identical across conditions of a dataset."""
    rng = np.random.RandomState(seed)
    idx = rng.permutation(len(students))
    n80, n90 = int(len(idx) * .8), int(len(idx) * .9)
    return idx[:n80], idx[n80:n90], idx[n90:]

def _json_safe(o):
    if isinstance(o, dict):  return {str(k): _json_safe(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_json_safe(v) for v in o]
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, np.ndarray): return _json_safe(o.tolist())
    return o

if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH) as f: _old_results = json.load(f)
    if _old_results.get("pipeline_version") == PIPELINE_VERSION:
        RESULTS = _old_results
        print(f"Resuming: {len(RESULTS.get('runs', {}))} compatible runs in {RESULTS_PATH}")
    else:
        print("Ignoring results from an earlier pipeline version; this correction requires a full rerun.")
        RESULTS = {"pipeline_version": PIPELINE_VERSION, "stats": {}, "shortcut": {}, "runs": {}}
else:
    RESULTS = {"pipeline_version": PIPELINE_VERSION, "stats": {}, "shortcut": {}, "runs": {}}

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

RESULTS["provenance"] = {"pipeline_version": PIPELINE_VERSION,
                           "python": sys.version.split()[0],
                           "platform": platform.platform(),
                           "data_sha256": {"ASSIST2009": sha256_file(RAW_A09),
                                           "ASSIST2017": sha256_file(RAW_A17)},
                           "config": dict(CFG)}

def save_results():
    with open(RESULTS_PATH, "w") as f:
        json.dump(_json_safe(RESULTS), f, indent=1)

def load_csv(path, **kw):
    for enc in ("utf-8", "latin-1", "ISO-8859-1"):
        try: return pd.read_csv(path, low_memory=False, encoding=enc, **kw)
        except UnicodeDecodeError: continue
    raise IOError(f"Cannot load {path}")

ASSIST2009 raw: /kaggle/input/datasets/youssefamk/2009assist/skill_builder_data.csv
ASSIST2017 raw: /kaggle/input/datasets/youssefamk/assist2017-uploaded-again/anonymized_full_release_competition_dataset.csv
Resuming: 45 compatible runs in paper_v3_out/paper_v3_results.json


In [3]:
# ================================================================
# 2. DATA PIPELINE — raw files → experimental conditions
#
# Every condition dataframe has the same schema, one row per model
# input step, in sequence order:
#   student_id, question_id, skill, correct, iid, pos
# `iid` = interaction id within student (all KC rows of one original
# interaction share it) — used for question-level fusion evaluation.
#
# ASSIST2009 conditions:
#   LEGACY : raw expanded rows as-is (KC-expansion + duplicate
#            logging bug) — what v1 (and naive pipelines) trained on
#   QL     : question level, one row per interaction (min KC id kept)
#   EXP    : clean KC-expansion of QL (one row per distinct KC,
#            consecutive) — pyKT-style expansion, bug removed
#   SHUF   : EXP with non-first KC rows scattered to non-adjacent
#            positions (order-breaking ablation)
# ASSIST2017 conditions: QL17, EXP17 (same builders)
# ================================================================

def load_a09_events(path):
    """Raw skill_builder → filtered event rows (chronological), + interaction id."""
    df = load_csv(path)
    rn = {}
    for c in df.columns:
        cl = c.strip().lower()
        if   cl == "user_id":     rn[c] = "student_id"
        elif cl == "problem_id":  rn[c] = "question_id"
        elif cl == "skill_id":    rn[c] = "skill"
        elif cl == "order_id":    rn[c] = "order_id"
        elif cl == "answer_type": rn[c] = "answer_type"
        elif cl == "original":    rn[c] = "original"
    df = df.rename(columns=rn)
    keep = [c for c in ["student_id","question_id","skill","correct",
                        "order_id","answer_type","original"] if c in df.columns]
    df = df[keep].copy()
    n0 = len(df)
    for c in ["student_id","question_id","correct","order_id","skill"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=["student_id","question_id","correct","order_id"])
    if "answer_type" in df.columns:
        df = df[df["answer_type"].astype(str).str.strip().str.lower() != "open_response"]
    if "original" in df.columns:
        dfo = df[pd.to_numeric(df["original"], errors="coerce") == 1]
        if len(dfo) > len(df) * 0.3: df = dfo
    df = df.dropna(subset=["skill"])
    df["correct"] = df["correct"].astype(int).clip(0, 1)
    for c in ["student_id","question_id","order_id","skill"]:
        df[c] = df[c].astype(np.int64)
    df = df.sort_values(["student_id","order_id"], kind="mergesort").reset_index(drop=True)
    # interaction id: rows sharing (student, raw order_id) are one interaction
    gid = df.groupby(["student_id","order_id"], sort=False).ngroup()
    df["iid"] = (gid - df.assign(_g=gid).groupby("student_id", sort=False)["_g"].transform("min")).astype(np.int64)
    print(f"  ASSIST2009: {n0:,} raw rows -> {len(df):,} filtered event rows, "
          f"{df['student_id'].nunique():,} students")
    return df[["student_id","question_id","skill","correct","iid"]]

def load_a17_events(path):
    """Raw ASSIST2017 → filtered event rows (chronological), + interaction id."""
    use = ["studentId","problemId","skill","correct","startTime"]
    df = load_csv(path, usecols=use)
    n0 = len(df)
    df = df.dropna(subset=use)
    df["correct"]   = pd.to_numeric(df["correct"], errors="coerce")
    df["startTime"] = pd.to_numeric(df["startTime"], errors="coerce")
    df = df.dropna(subset=["correct","startTime"])
    df["correct"] = df["correct"].astype(int).clip(0, 1)
    df["skill"] = df["skill"].astype(str).str.strip()
    df = df[(df["skill"].str.len() > 0) & (df["skill"].str.lower() != "nan")]
    df = df.rename(columns={"studentId":"student_id","problemId":"question_id"})
    df = df.sort_values(["student_id","startTime","question_id"], kind="mergesort").reset_index(drop=True)
    # A17 has no reliable interaction identifier. Timestamp/problem ties can
    # contain conflicting outcomes (distinct attempts recorded at the same
    # timestamp resolution), so collapsing such ties is scientifically unsafe.
    # Preserve every source row as one chronological interaction; mergesort
    # keeps source order deterministic within timestamp ties.
    tie = df.groupby(["student_id", "startTime", "question_id"], sort=False)
    n_tied = int((tie.size() > 1).sum())
    n_conflict = int((tie["correct"].nunique() > 1).sum())
    df["iid"] = df.groupby("student_id", sort=False).cumcount().astype(np.int64)
    print(f"  ASSIST2017: {n0:,} raw rows -> {len(df):,} filtered event rows, "
          f"{df['student_id'].nunique():,} students; "
          f"{n_tied:,} timestamp/problem ties retained as separate rows "
          f"({n_conflict:,} with conflicting labels)")
    return df[["student_id","question_id","skill","correct","iid"]]

def _with_pos(df):
    df = df.reset_index(drop=True)
    df["pos"] = df.groupby("student_id", sort=False).cumcount()
    return df

def build_conditions(events, min_seq=MIN_SEQ, tag=""):
    """events (chronological, with iid) → dict of condition dataframes + stats."""
    legacy = events.copy()
    chk = legacy.groupby(["student_id", "iid"], sort=False).agg(
        n_question=("question_id", "nunique"), n_label=("correct", "nunique"))
    n_bad_question = int((chk["n_question"] > 1).sum())
    n_bad_label = int((chk["n_label"] > 1).sum())
    if n_bad_question or n_bad_label:
        raise ValueError(f"{tag}: ambiguous interaction IDs: {n_bad_question} question conflicts, {n_bad_label} label conflicts")

    # remove duplicate interaction-KC records: same interaction and KC listed twice
    dedup = legacy.drop_duplicates(subset=["student_id","iid","skill"]).copy()
    n_bug = len(legacy) - len(dedup)

    # question level: one row per interaction, distinct KCs kept as a tuple
    ql = (dedup.groupby(["student_id","iid"], sort=False)
               .agg(question_id=("question_id","first"),
                    correct=("correct","first"),
                    kcs=("skill", lambda s: tuple(sorted(pd.unique(s)))))
               .reset_index())
    ql = ql.sort_values(["student_id","iid"], kind="mergesort").reset_index(drop=True)

    # same student set for every condition
    cnt = ql.groupby("student_id").size()
    students = sorted(cnt[cnt >= min_seq].index)
    sset = set(students)
    ql     = ql[ql["student_id"].isin(sset)].reset_index(drop=True)
    legacy = legacy[legacy["student_id"].isin(sset)].reset_index(drop=True)

    # clean KC-expansion: one row per distinct KC, consecutive
    exp = (ql[["student_id","question_id","correct","iid","kcs"]]
           .explode("kcs").rename(columns={"kcs":"skill"}).reset_index(drop=True))

    ql = ql.copy()
    ql["n_kc"]  = ql["kcs"].str.len()
    ql["skill"] = ql["kcs"].str[0]

    cols = ["student_id","question_id","skill","correct","iid"]
    conds = {"LEGACY": _with_pos(legacy[cols]),
             "QL":     _with_pos(ql[cols]),
             "EXP":    _with_pos(exp[cols])}

    stats = {
        "students": len(students),
        "rows_legacy": len(legacy), "rows_bug_duplicates": int(n_bug),
        "rows_exp": len(exp), "rows_ql": len(ql),
        "interaction_question_conflicts": n_bad_question, "interaction_label_conflicts": n_bad_label,
        "exp_equals_ql": len(exp) == len(ql),   # no multi-KC interaction -> EXP is literally QL
        "avg_kc_per_interaction": float(ql["n_kc"].mean()),
        "multi_kc_pct": float((ql["n_kc"] > 1).mean() * 100),
        "kc_distribution": {str(k): int(v) for k, v in
                            ql["n_kc"].value_counts().sort_index().items()},
    }
    print(f"  [{tag}] students={stats['students']:,}  interactions(QL)={stats['rows_ql']:,}  "
          f"clean-EXP rows={stats['rows_exp']:,}  LEGACY rows={stats['rows_legacy']:,}  "
          f"(bug duplicates: {n_bug:,})")
    print(f"  [{tag}] avg KCs/interaction={stats['avg_kc_per_interaction']:.3f}  "
          f"multi-KC={stats['multi_kc_pct']:.2f}%")
    return conds, students, stats

def make_shuffled(exp_df, seed=MAIN_SEED):
    """Order-breaking ablation: scatter non-first KC rows to non-adjacent positions."""
    rng = np.random.RandomState(seed)
    cols = ["student_id","question_id","skill","correct","iid"]
    parts, forced, n_extra = [], 0, 0
    for uid, g in exp_df.groupby("student_id", sort=False):
        arr  = g[cols].to_numpy()
        iids = g["iid"].to_numpy()
        first = np.r_[True, iids[1:] != iids[:-1]]
        base   = [arr[i] for i in np.where(first)[0]]
        extras = [arr[i] for i in np.where(~first)[0]]
        n_extra += len(extras)
        rng.shuffle(extras)
        for row in extras:
            qid, ok = row[1], False
            for _ in range(64):
                p = rng.randint(0, len(base) + 1)
                if (p == 0 or base[p-1][1] != qid) and (p == len(base) or base[p][1] != qid):
                    base.insert(p, row); ok = True; break
            if not ok:
                base.insert(rng.randint(0, len(base) + 1), row); forced += 1
        parts.append(pd.DataFrame(base, columns=cols))
    out = _with_pos(pd.concat(parts, ignore_index=True))
    same_q = float((out["question_id"] == out.groupby("student_id")["question_id"].shift(1)).mean())
    print(f"  [SHUF] scattered {n_extra:,} KC rows ({forced} forced placements); "
          f"same-Q consecutive rate after shuffling: {same_q*100:.2f}%")
    return out

print("Building ASSIST2009 conditions ...")
A09_CONDS, A09_STUDENTS, A09_STATS = build_conditions(load_a09_events(RAW_A09), tag="A09")
A09_CONDS["SHUF"] = make_shuffled(A09_CONDS["EXP"])

print("\nBuilding ASSIST2017 conditions ...")
A17_CONDS, A17_STUDENTS, A17_STATS = build_conditions(load_a17_events(RAW_A17), tag="A17")

Building ASSIST2009 conditions ...
  ASSIST2009: 525,534 raw rows -> 433,153 filtered event rows, 4,163 students
  [A09] students=3,841  interactions(QL)=258,888  clean-EXP rows=311,630  LEGACY rows=432,559  (bug duplicates: 120,929)
  [A09] avg KCs/interaction=1.204  multi-KC=17.35%
  [SHUF] scattered 52,742 KC rows (0 forced placements); same-Q consecutive rate after shuffling: 0.01%

Building ASSIST2017 conditions ...
  ASSIST2017: 942,816 raw rows -> 942,816 filtered event rows, 1,709 students; 5,914 timestamp/problem ties retained as separate rows (1,797 with conflicting labels)
  [A17] students=1,708  interactions(QL)=942,814  clean-EXP rows=942,814  LEGACY rows=942,814  (bug duplicates: 0)
  [A17] avg KCs/interaction=1.000  multi-KC=0.00%


In [4]:
# ================================================================
# 3. SECTION A — dataset statistics + duplicate decomposition
# ================================================================
def consec_stats(df):
    qp = df.groupby("student_id", sort=False)["question_id"].shift(1)
    rp = df.groupby("student_id", sort=False)["correct"].shift(1)
    same_q = (df["question_id"] == qp)
    p_q = float(same_q.mean())
    p_l = float((df.loc[same_q, "correct"].values == rp[same_q].values).mean()) if same_q.any() else float("nan")
    return p_q, p_l

if RUN["A"]:
    print("=" * 65); print("SECTION A: Dataset statistics"); print("=" * 65)
    for ds, conds, st in [("A09", A09_CONDS, A09_STATS), ("A17", A17_CONDS, A17_STATS)]:
        st = dict(st)
        for cname, cdf in conds.items():
            p_q, p_l = consec_stats(cdf)
            st[f"consec_{cname}"] = {"rows": len(cdf),
                                     "p_same_q_pct": round(p_q * 100, 2),
                                     "p_same_label_given_same_q_pct": round(p_l * 100, 2) if not math.isnan(p_l) else None}
            print(f"  [{ds}/{cname:6s}] rows={len(cdf):>9,}  P(same-Q consec)={p_q*100:6.2f}%  "
                  f"P(same label | same-Q)={'n/a' if math.isnan(p_l) else f'{p_l*100:.2f}%'}")
        # how much of LEGACY's duplication is NOT explained by KC-expansion
        st["expansion_factor_clean"]  = st["rows_exp"] / st["rows_ql"]
        st["expansion_factor_legacy"] = st["rows_legacy"] / st["rows_ql"]
        print(f"  [{ds}] duplication factor: clean KC-expansion x{st['expansion_factor_clean']:.3f}, "
              f"legacy raw x{st['expansion_factor_legacy']:.3f} "
              f"(gap = duplicate interaction-KC records)")
        RESULTS["stats"][ds] = st
    save_results()

    # ── Figure 1: structure + decomposition ─────────────────────
    s9 = RESULTS["stats"]["A09"]
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))

    kd = s9["kc_distribution"]
    ks = [int(k) for k in kd]; vs = [v / sum(kd.values()) * 100 for v in kd.values()]
    axes[0].bar(ks, vs, color="#2196F3")
    for k, v in zip(ks, vs):
        axes[0].text(k, v + 0.5, f"{v:.1f}%", ha="center", fontsize=9)
    axes[0].set_xlabel("Distinct KCs per interaction"); axes[0].set_ylabel("% of interactions")
    axes[0].set_title(f"ASSIST2009 KC distribution\n({s9['multi_kc_pct']:.1f}% multi-KC, "
                      f"avg {s9['avg_kc_per_interaction']:.3f})")

    labels = ["QL\n(interactions)", "EXP\n(clean KC-exp.)", "LEGACY\n(raw file)"]
    rows   = [s9["rows_ql"], s9["rows_exp"], s9["rows_legacy"]]
    base   = s9["rows_ql"]
    axes[1].bar(labels, [base]*3, color="#1976D2", label="question-level rows")
    axes[1].bar(labels, [0, s9["rows_exp"]-base, s9["rows_exp"]-base], bottom=[base]*3,
                color="#FF9800", label="+ KC-expansion rows")
    axes[1].bar(labels, [0, 0, s9["rows_legacy"]-s9["rows_exp"]],
                bottom=[base, s9["rows_exp"], s9["rows_exp"]],
                color="#F44336", label="+ duplicate interaction-KC records")
    for x, r in zip(range(3), rows):
        axes[1].text(x, r + base*0.01, f"{r:,}", ha="center", fontsize=10, fontweight="bold")
    axes[1].legend(fontsize=9); axes[1].set_ylabel("rows")
    axes[1].set_title("ASSIST2009 row decomposition")

    names, pq, pl = [], [], []
    for ds, cn in [("A09","QL"), ("A09","EXP"), ("A09","LEGACY"), ("A09","SHUF"), ("A17","EXP")]:
        c = RESULTS["stats"][ds][f"consec_{cn}"]
        names.append(f"{'A09' if ds=='A09' else 'A17'}\n{cn}")
        pq.append(c["p_same_q_pct"]); pl.append(c["p_same_label_given_same_q_pct"] or 0)
    x = np.arange(len(names)); w = 0.38
    axes[2].bar(x - w/2, pq, w, color="#2196F3", label="P(same-Q consecutive)")
    axes[2].bar(x + w/2, pl, w, color="#FF5722", label="P(same label | same-Q)")
    for xi, (a, b) in enumerate(zip(pq, pl)):
        if a > 0: axes[2].text(xi - w/2, a + 1, f"{a:.1f}", ha="center", fontsize=8)
        if b > 0: axes[2].text(xi + w/2, b + 1, f"{b:.1f}", ha="center", fontsize=8)
    axes[2].set_xticks(x); axes[2].set_xticklabels(names, fontsize=9)
    axes[2].set_ylim(0, 115); axes[2].legend(fontsize=9)
    axes[2].set_title("Consecutive-duplicate structure\n(100% label consistency = artificial copy)")

    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/fig1_dataset_structure.png", dpi=150, bbox_inches="tight")
    plt.close()
    print(f"\nSaved: {OUT_DIR}/fig1_dataset_structure.png")

SECTION A: Dataset statistics
  [A09/LEGACY] rows=  432,559  P(same-Q consec)= 40.16%  P(same label | same-Q)=100.00%
  [A09/QL    ] rows=  258,888  P(same-Q consec)=  0.01%  P(same label | same-Q)=88.00%
  [A09/EXP   ] rows=  311,630  P(same-Q consec)= 16.93%  P(same label | same-Q)=99.99%
  [A09/SHUF  ] rows=  311,630  P(same-Q consec)=  0.01%  P(same label | same-Q)=86.36%
  [A09] duplication factor: clean KC-expansion x1.204, legacy raw x1.671 (gap = duplicate interaction-KC records)
  [A17/LEGACY] rows=  942,814  P(same-Q consec)= 52.35%  P(same label | same-Q)=68.17%
  [A17/QL    ] rows=  942,814  P(same-Q consec)= 52.35%  P(same label | same-Q)=68.17%
  [A17/EXP   ] rows=  942,814  P(same-Q consec)= 52.35%  P(same label | same-Q)=68.17%
  [A17] duplication factor: clean KC-expansion x1.000, legacy raw x1.000 (gap = duplicate interaction-KC records)

Saved: paper_v3_out/fig1_dataset_structure.png


In [5]:
# ================================================================
# 4. SECTION B — zero-parameter shortcut baseline  (v3)
#    Rule: for each target position t (t >= 1 within a student), look
#    back up to `lookback` positions in the same student; if any shares
#    the question id, predict that row's label ("fire"). Otherwise
#    predict the student's running mean over all previously observed rows.
#    A 0.5 cold-start fallback is defined but is not reached because row 0
#    is context-only and evaluation begins at row 1.
#    Evaluated on each seed's TEST-split students; mean ± std reported.
# ================================================================
from sklearn.metrics import roc_auc_score

def compute_auc(yt, yp):
    yt = np.asarray(yt, int); yp = np.asarray(yp, float)
    if yt.size == 0 or len(np.unique(yt)) < 2: return float("nan")
    return float(roc_auc_score(yt, yp))

def shortcut_auc(df, students_subset=None, lookback=5):
    if students_subset is not None:
        df = df[df["student_id"].isin(set(students_subset))]
    yt, yp, fires = [], [], 0
    for uid, g in df.groupby("student_id", sort=False):
        q = g["question_id"].to_numpy(); r = g["correct"].to_numpy()
        rc, rt = int(r[0]), 1
        for i in range(1, len(q)):
            pred = None
            for j in range(i - 1, max(-1, i - 1 - lookback), -1):
                if q[j] == q[i]:
                    pred = float(r[j]); fires += 1; break
            if pred is None:
                pred = rc / rt if rt else 0.5
            yt.append(int(r[i])); yp.append(pred)
            rc += int(r[i]); rt += 1
    return compute_auc(yt, yp), fires / max(len(yt), 1)

if RUN["B"]:
    print("=" * 65); print("SECTION B: Shortcut baseline (zero parameters)"); print("=" * 65)
    cases = [("A09/QL", A09_CONDS["QL"], A09_STUDENTS),
             ("A09/EXP", A09_CONDS["EXP"], A09_STUDENTS),
             ("A09/LEGACY", A09_CONDS["LEGACY"], A09_STUDENTS),
             ("A09/SHUF", A09_CONDS["SHUF"], A09_STUDENTS),
             ("A17/QL", A17_CONDS["QL"], A17_STUDENTS)]
    if not A17_STATS["exp_equals_ql"]:
        cases.append(("A17/EXP", A17_CONDS["EXP"], A17_STUDENTS))
    else:
        print("  [A17/EXP] skipped — identical to A17/QL (zero multi-KC interactions)")
    for name, df, studs in cases:
        if name in RESULTS["shortcut"]:
            r = RESULTS["shortcut"][name]
            print(f"  [{name}] cached: AUC={np.mean(r['auc_per_seed']):.4f}"); continue
        t0 = time.time(); aucs, frs = [], []
        for sd_ in SEEDS:
            _, _, te_i = split_students(studs, sd_)
            te_students = [studs[i] for i in te_i]
            a, fr = shortcut_auc(df, te_students)
            aucs.append(a); frs.append(fr)
        RESULTS["shortcut"][name] = {
            "auc_per_seed": aucs, "fire_per_seed": frs,
            "auc_mean": float(np.mean(aucs)), "auc_std": float(np.std(aucs, ddof=1)),
            "fire_mean": float(np.mean(frs))}
        save_results()
        print(f"  [{name:10s}] AUC={np.mean(aucs):.4f}±{np.std(aucs, ddof=1):.4f}  "
              f"fire={np.mean(frs)*100:5.1f}%   ({time.time()-t0:.0f}s)")

    # ── Figure 2 ────────────────────────────────────────────────
    names = [n for n, _, _ in cases if n in RESULTS["shortcut"]]
    aucs  = [RESULTS["shortcut"][n]["auc_mean"] for n in names]
    errs  = [RESULTS["shortcut"][n]["auc_std"] for n in names]
    frs   = [RESULTS["shortcut"][n]["fire_mean"] * 100 for n in names]
    fig, ax1 = plt.subplots(figsize=(10, 5))
    x = np.arange(len(names))
    colors = ["#4CAF50", "#F44336", "#B71C1C", "#FF9800", "#4CAF50", "#4CAF50"][:len(names)]
    bars = ax1.bar(x, aucs, 0.5, yerr=errs, capsize=4, color=colors, alpha=0.9)
    ax1.axhline(0.5, color="gray", ls="--", lw=0.8)
    for b, a in zip(bars, aucs):
        ax1.text(b.get_x() + b.get_width()/2, b.get_height() + 0.006, f"{a:.3f}",
                 ha="center", fontsize=10, fontweight="bold")
    ax1.set_xticks(x); ax1.set_xticklabels([n.replace("/", "\n") for n in names], fontsize=9)
    ax1.set_ylabel("AUC"); ax1.set_ylim(0.45, 1.0)
    ax1.set_title("Shortcut baseline: copy most recent same-question label (0 parameters)\n"
                  "test-split students, mean ± std over 5 seeds")
    ax2 = ax1.twinx()
    ax2.plot(x, frs, "ko--", ms=6)
    for xi, fr in zip(x, frs):
        ax2.annotate(f"{fr:.1f}%", (xi, fr), textcoords="offset points", xytext=(8, 4), fontsize=8)
    ax2.set_ylabel("fire rate (%)")
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/fig2_shortcut_baseline.png", dpi=150, bbox_inches="tight")
    plt.close()
    print(f"\nSaved: {OUT_DIR}/fig2_shortcut_baseline.png")

SECTION B: Shortcut baseline (zero parameters)
  [A17/EXP] skipped — identical to A17/QL (zero multi-KC interactions)
  [A09/QL] cached: AUC=0.6810
  [A09/EXP] cached: AUC=0.7791
  [A09/LEGACY] cached: AUC=0.8652
  [A09/SHUF] cached: AUC=0.6931
  [A17/QL] cached: AUC=0.5997

Saved: paper_v3_out/fig2_shortcut_baseline.png


In [6]:
# ================================================================
# 5. MODELS + TRAINING INFRASTRUCTURE  (v3)
#
# Windows are built over INTERACTIONS (never split), so all conditions
# see the same history coverage per window.
#
# Evaluation protocols:
#   standard   : teacher-forced row inputs (leaky on expanded data);
#                yields row AUC and fused (interaction-averaged) AUC.
#   all-in-one : every KC of an interaction is predicted independently
#                with history ending at the previous interaction, while
#                PAST interactions keep their real responses in context
#                (faithful history, as in pyKT's protocol). Exact for all
#                three models:
#                - attention models: each response is embedded on its
#                  source row, attention is strictly causal, and every
#                  same-interaction key/value path is banned. Queries
#                  carry no response; all prior interactions remain.
#                - DKT: every sibling is read directly from the recurrent
#                  state at the end of the previous interaction.
# Model selection: validation all-in-one fused AUC for every run
# (coincides with row AUC on question-level data) — no leaky selection.
# ================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
RESULTS["provenance"]["torch"] = torch.__version__

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    print("WARNING: no GPU found — training will be extremely slow.")

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def deterministic_log_bin(arr, n_bins):
    """History-only count bins; no quantile is fitted on validation/test users."""
    arr = np.clip(np.asarray(arr, float), 0, None)
    return np.minimum(np.floor(np.log2(arr + 1)).astype(np.int64), n_bins - 1)

def prepare_condition(df, student_index):
    """Encode ids; sequences keyed by encoded student.
    Array columns: [u, q, s, r, att_bin, iid]."""
    df = df.copy()
    df["u"] = df["student_id"].map(student_index).astype(np.int64)
    df["q"] = pd.factorize(df["question_id"].astype(str))[0].astype(np.int64)
    df["s"] = pd.factorize(df["skill"].astype(str))[0].astype(np.int64)
    nq, ns = int(df["q"].max()) + 1, int(df["s"].max()) + 1
    df = df.sort_values(["u", "pos"], kind="mergesort").reset_index(drop=True)
    # Opportunity counts advance once per distinct interaction-KC, not once
    # per duplicate record; every duplicate of an interaction-KC shares a bin.
    opp = df[["u", "iid", "s"]].drop_duplicates().copy()
    opp["att_count"] = opp.groupby(["u", "s"], sort=False).cumcount()
    df = df.merge(opp, on=["u", "iid", "s"], how="left", sort=False, validate="many_to_one")
    df = df.sort_values(["u", "pos"], kind="mergesort").reset_index(drop=True)
    att = deterministic_log_bin(df["att_count"].to_numpy(float), CFG["n_att_bins"])
    arr = np.stack([df["u"].to_numpy(np.int32), df["q"].to_numpy(np.int32),
                    df["s"].to_numpy(np.int32), df["correct"].to_numpy(np.int32),
                    att.astype(np.int32), df["iid"].to_numpy(np.int32)], axis=1)
    u_arr  = arr[:, 0]
    bounds = np.where(np.diff(u_arr))[0] + 1
    seqs = {int(uid): seg for uid, seg in zip(u_arr[np.r_[0, bounds]], np.split(arr, bounds))}
    return seqs, nq, ns

def build_window_catalog(conditions, student_index, tag):
    """Build shared windows from QL interaction IDs before model-row materialisation.
    The row budget is computed from the maximum per-interaction row count over
    all conditions, so any shortening is identical and no interaction is split."""
    ql = conditions["QL"][["student_id", "iid", "pos"]].copy()
    ql["u"] = ql["student_id"].map(student_index).astype(np.int64)
    ordered = {int(u): g.sort_values("pos")["iid"].astype(int).tolist()
               for u, g in ql.groupby("u", sort=False)}
    max_cost = {}
    for cdf in conditions.values():
        z = cdf[["student_id", "iid"]].copy()
        z["u"] = z["student_id"].map(student_index).astype(np.int64)
        for (u, iid), n in z.groupby(["u", "iid"], sort=False).size().items():
            key = (int(u), int(iid)); max_cost[key] = max(max_cost.get(key, 0), int(n))

    def make_specs(stride):
        out, shortened, shared_rows = {}, 0, []
        for u, iids in ordered.items():
            costs = np.array([max_cost[(u, iid)] for iid in iids], dtype=np.int64)
            if costs.max(initial=0) > CFG["max_rows"]:
                raise ValueError(f"{tag}: one interaction exceeds max_rows; increase the shared budget")
            pref = np.r_[0, np.cumsum(costs)]; n_int = len(iids); s0 = 0
            while s0 < n_int:
                nominal = min(s0 + CFG["max_inter"], n_int); e0 = nominal
                while e0 - s0 > 1 and pref[e0] - pref[s0] > CFG["max_rows"]:
                    e0 -= 1
                if pref[e0] - pref[s0] > CFG["max_rows"]:
                    raise ValueError(f"{tag}: shared window budget cannot hold interaction {iids[s0]}")
                if e0 - s0 >= 2:
                    out.setdefault(u, []).append(tuple(iids[s0:e0]))
                    shared_rows.append(int(pref[e0] - pref[s0]))
                    shortened += int(e0 < nominal)
                if e0 >= n_int: break
                s0 += min(stride, max(1, e0 - s0))
        audit = {"windows": int(sum(map(len, out.values()))),
                 "shortened_by_shared_row_budget": int(shortened),
                 "max_shared_rows": int(max(shared_rows, default=0)),
                 "max_interactions": int(max((len(w) for ws in out.values() for w in ws), default=0))}
        return out, audit

    tr, tr_a = make_specs(CFG["train_stride_inter"])
    ev, ev_a = make_specs(CFG["eval_stride_inter"])
    print(f"  [{tag}] shared windows: train={tr_a}, eval={ev_a}")
    return {"train": tr, "eval": ev, "audit": {"train": tr_a, "eval": ev_a}}

class KTDataset(Dataset):
    """Materialise a condition using shared original-interaction windows."""
    def __init__(self, seqs, specs, users):
        self.s = []
        for u in users:
            u = int(u)
            if u not in seqs: continue
            seq = seqs[u]
            for spec in specs.get(u, []):
                w = seq[np.isin(seq[:, 5], np.asarray(spec, dtype=seq.dtype))]
                if set(map(int, np.unique(w[:, 5]))) != set(map(int, spec)):
                    raise AssertionError(f"condition is missing interaction(s) for user {u}")
                target = (w[:, 5] != int(spec[0])).astype(np.int64)
                self.s.append(tuple(w[:, c].astype(np.int64) for c in range(6)) + (target,))
    def __len__(self): return len(self.s)
    def __getitem__(self, i): return self.s[i]

def make_collate(pad_q, pad_s):
    def cfn(batch):
        L = max(len(x[0]) for x in batch); B = len(batch)
        u_ = np.zeros((B, L), np.int64); q_ = np.full((B, L), pad_q, np.int64)
        s_ = np.full((B, L), pad_s, np.int64); r_ = np.zeros((B, L), np.int64)
        a_ = np.zeros((B, L), np.int64); i_ = np.full((B, L), -1, np.int64)
        m_ = np.zeros((B, L), np.float32); z_ = np.zeros((B, L), np.bool_)
        for i, (u, q, s, r, a, ii, z) in enumerate(batch):
            l = len(q)
            u_[i, :l] = u; q_[i, :l] = q; s_[i, :l] = s
            r_[i, :l] = r; a_[i, :l] = a; i_[i, :l] = ii; m_[i, :l] = 1.; z_[i, :l] = z
        t = lambda x, d: torch.tensor(x, dtype=d)
        return (t(u_, torch.long), t(q_, torch.long), t(s_, torch.long),
                t(r_, torch.long), t(a_, torch.long), t(i_, torch.long),
                t(m_, torch.float32), t(z_, torch.bool))
    return cfn

def make_loaders(seqs, students, nq, ns, seed, window_catalog):
    tr_i, va_i, te_i = split_students(students, seed)
    if torch.cuda.is_available():
        ng = torch.cuda.device_count()
        vr = torch.cuda.get_device_properties(0).total_memory / 1e9
        bs = (64 if vr >= 14 else 32) * ng
    else:
        bs = 16
    cfn = make_collate(nq, ns)
    nw  = 2 if torch.cuda.is_available() else 0
    def mk(users, shuffle):
        specs = window_catalog["train" if shuffle else "eval"]
        return DataLoader(KTDataset(seqs, specs, users),
                          bs, shuffle=shuffle, num_workers=nw,
                          pin_memory=torch.cuda.is_available(), collate_fn=cfn)
    return mk(tr_i, True), mk(va_i, False), mk(te_i, False)

# ── models: forward(q, s, r, a, mask, ban=None) ────────────────
# ban: (B, L, L) bool, True = attention prohibited (same-interaction
# rows). Attention is already strictly causal, including the diagonal.
class DKT(nn.Module):
    def __init__(self, nq, ns, d=128, dr=0.1, **kw):
        super().__init__()
        self.emb = nn.Embedding(2 * (ns + 1), d)
        self.gru  = nn.GRU(d, d, batch_first=True)
        self.drop = nn.Dropout(dr)
        self.head = nn.Linear(d, ns + 1)
        for m in self.modules():
            if isinstance(m, nn.Linear): nn.init.xavier_uniform_(m.weight)
            elif isinstance(m, nn.Embedding): nn.init.normal_(m.weight, std=0.01)
    def _readout(self, h, s):
        return self.head(self.drop(h)).gather(-1, s.unsqueeze(-1)).squeeze(-1)
    def forward(self, q, s, r, a, mask, ban=None):
        # Canonical DKT timing: update with (skill_t, response_t), then shift
        # the hidden states so the prediction for t uses history through t-1.
        x = self.emb((s * 2 + r).clamp(max=self.emb.num_embeddings - 1))
        h_after, _ = self.gru(self.drop(x))
        h_before = torch.zeros_like(h_after); h_before[:, 1:] = h_after[:, :-1]
        return self._readout(h_before, s)
    def allinone_logits(self, q, s, r, a, mask, ii):
        """Every sibling is read from the same state at the previous
        interaction boundary; no target-interaction update precedes prediction."""
        B, L = r.shape; d = self.gru.hidden_size
        x = self.emb((s * 2 + r).clamp(max=self.emb.num_embeddings - 1))
        h, _ = self.gru(self.drop(x))                     # observed past interactions
        fr = torch.ones_like(ii, dtype=torch.bool)
        fr[:, 1:] = ii[:, 1:] != ii[:, :-1]                # first row of interaction
        idx = torch.arange(L, device=r.device).unsqueeze(0).expand(B, L)
        first_idx = torch.cummax(torch.where(fr, idx, torch.zeros_like(idx)), 1).values
        prev_end = first_idx - 1                           # -1 -> window-initial state
        h_prev = h.gather(1, prev_end.clamp(min=0).unsqueeze(-1).expand(B, L, d))
        h_prev = torch.where((prev_end >= 0).unsqueeze(-1), h_prev, torch.zeros_like(h_prev))
        return self._readout(h_prev, s)

class _AKTAttn(nn.Module):
    def __init__(self, d, nh, dr):
        super().__init__()
        self.H = nh; self.dk = d // nh
        self.Wq = nn.Linear(d, d); self.Wk = nn.Linear(d, d)
        self.Wv = nn.Linear(d, d); self.Wo = nn.Linear(d, d); self.drop = nn.Dropout(dr)
        self.gamma = nn.Parameter(torch.full((nh,), math.log(math.e - 1)))
    def forward(self, Q, K, V, mask=None, ban=None):
        B, L, D = Q.shape; H, dk = self.H, self.dk
        sp = lambda x: x.view(B, L, H, dk).transpose(1, 2)
        Qp, Kp, Vp = sp(self.Wq(Q)), sp(self.Wk(K)), sp(self.Wv(V))
        sc = (Qp @ Kp.transpose(-2, -1)) / math.sqrt(dk)
        t  = torch.arange(L, device=Q.device, dtype=torch.float)
        dist = (t.unsqueeze(1) - t.unsqueeze(0)).clamp(0)
        sc = sc * torch.exp(-F.softplus(self.gamma).view(1, H, 1, 1) * dist.unsqueeze(0).unsqueeze(0))
        cm = torch.triu(torch.ones(L, L, device=Q.device, dtype=torch.bool), 0)
        sc = sc.masked_fill(cm.unsqueeze(0).unsqueeze(0), float("-inf"))
        if mask is not None: sc = sc.masked_fill((mask == 0).unsqueeze(1).unsqueeze(2), float("-inf"))
        if ban is not None:  sc = sc.masked_fill(ban.unsqueeze(1), float("-inf"))
        at = torch.nan_to_num(F.softmax(sc, -1), nan=0.)
        return self.Wo((self.drop(at) @ Vp).transpose(1, 2).contiguous().view(B, L, D))

class _PlainAttn(nn.Module):
    def __init__(self, d, nh, dr):
        super().__init__()
        self.H = nh; self.dk = d // nh
        self.Wq = nn.Linear(d, d); self.Wk = nn.Linear(d, d)
        self.Wv = nn.Linear(d, d); self.Wo = nn.Linear(d, d); self.drop = nn.Dropout(dr)
    def forward(self, Q, K, V, mask=None, ban=None):
        B, L, D = Q.shape; H, dk = self.H, self.dk
        sp = lambda x: x.view(B, L, H, dk).transpose(1, 2)
        Qp, Kp, Vp = sp(self.Wq(Q)), sp(self.Wk(K)), sp(self.Wv(V))
        sc = (Qp @ Kp.transpose(-2, -1)) / math.sqrt(dk)
        cm = torch.triu(torch.ones(L, L, device=Q.device, dtype=torch.bool), 0)
        sc = sc.masked_fill(cm.unsqueeze(0).unsqueeze(0), float("-inf"))
        if mask is not None: sc = sc.masked_fill((mask == 0).unsqueeze(1).unsqueeze(2), float("-inf"))
        if ban is not None:  sc = sc.masked_fill(ban.unsqueeze(1), float("-inf"))
        at = torch.nan_to_num(F.softmax(sc, -1), nan=0.)
        return self.Wo((self.drop(at) @ Vp).transpose(1, 2).contiguous().view(B, L, D))

def _blk(Attn, d, nh, dr):
    class Blk(nn.Module):
        def __init__(self):
            super().__init__()
            self.attn = Attn(d, nh, dr)
            self.ff = nn.Sequential(nn.Linear(d, d * 4), nn.GELU(), nn.Dropout(dr), nn.Linear(d * 4, d))
            self.n1 = nn.LayerNorm(d); self.n2 = nn.LayerNorm(d); self.drop = nn.Dropout(dr)
        def forward(self, q, kv, mask=None, ban=None):
            x = q + self.drop(self.attn(self.n1(q), self.n1(kv), self.n1(kv), mask, ban))
            return x + self.drop(self.ff(self.n2(x)))
    return Blk()

class AKTR(nn.Module):
    def __init__(self, nq, ns, d=128, nh=8, nl=2, dr=0.1, n_att=16):
        super().__init__()
        self.kc = nn.Embedding(ns + 1, d); self.dd = nn.Embedding(ns + 1, d)
        self.mu = nn.Embedding(nq + 1, 1); self.re = nn.Embedding(2, d)
        self.ae = nn.Embedding(n_att, d)
        self.q_enc = nn.ModuleList([_blk(_AKTAttn, d, nh, dr) for _ in range(nl)])
        self.k_enc = nn.ModuleList([_blk(_AKTAttn, d, nh, dr) for _ in range(nl)])
        self.ret   = nn.ModuleList([_blk(_AKTAttn, d, nh, dr) for _ in range(nl)])
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Linear(d, d // 2), nn.GELU(), nn.Dropout(dr), nn.Linear(d // 2, 1))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding): nn.init.normal_(m.weight, std=0.01)
        nn.init.zeros_(self.mu.weight)
    def forward(self, q, s, r, a, mask, ban=None):
        xq = self.kc(s) + self.mu(q) * self.dd(s); xi = xq + self.re(r) + self.ae(a)
        h = xq
        for blk in self.q_enc: h = blk(h, h, mask, ban)
        k = xi
        for blk in self.k_enc: k = blk(k, k, mask, ban)
        for blk in self.ret:   h = blk(h, k, mask, ban)
        return self.head(self.norm(h)).squeeze(-1)

class SimpleKT(nn.Module):
    def __init__(self, nq, ns, d=128, nh=8, nl=2, dr=0.1, n_att=16):
        super().__init__()
        self.kc = nn.Embedding(ns + 1, d); self.var = nn.Embedding(ns + 1, d)
        self.dif = nn.Embedding(nq + 1, d); self.res = nn.Embedding(2, d)
        self.ae = nn.Embedding(n_att, d)
        self.blocks = nn.ModuleList([_blk(_PlainAttn, d, nh, dr) for _ in range(nl)])
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Linear(d, d // 2), nn.GELU(), nn.Dropout(dr), nn.Linear(d // 2, 1))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding): nn.init.normal_(m.weight, std=0.01)
        nn.init.zeros_(self.dif.weight)
    def forward(self, q, s, r, a, mask, ban=None):
        x = self.kc(s) + self.dif(q) * self.var(s); y = x + self.res(r) + self.ae(a)
        h = x
        for blk in self.blocks: h = blk(h, y, mask, ban)
        return self.head(self.norm(h)).squeeze(-1)

def build_model(name, nq, ns):
    d, nh, nl = CFG["d_model"], CFG["n_heads"], CFG["n_layers"]
    dr, na = CFG["dropout"], CFG["n_att_bins"]
    if name == "DKT":      return DKT(nq, ns, d, dr)
    if name == "AKT-R":    return AKTR(nq, ns, d, nh, nl, dr, na)
    if name == "simpleKT": return SimpleKT(nq, ns, d, nh, nl, dr, na)
    raise ValueError(name)

class EMA:
    def __init__(self, m, d):
        self.d = d
        self.s = {n: p.data.clone() for n, p in m.named_parameters() if p.requires_grad}
        self.b = {}
    @torch.no_grad()
    def update(self, m):
        for n, p in m.named_parameters():
            if p.requires_grad: self.s[n].mul_(self.d).add_(p.data, alpha=1 - self.d)
    def apply(self, m):
        self.b = {}
        for n, p in m.named_parameters():
            if p.requires_grad: self.b[n] = p.data.clone(); p.data.copy_(self.s[n])
    def restore(self, m):
        for n, p in m.named_parameters():
            if p.requires_grad: p.data.copy_(self.b[n])
        self.b = {}

def _scaler(en):
    try: return torch.amp.GradScaler("cuda", enabled=en)
    except Exception: return torch.cuda.amp.GradScaler(enabled=en)
def _autocast(en):
    try: return torch.amp.autocast("cuda", enabled=en)
    except Exception: return torch.cuda.amp.autocast(enabled=en)

def cosine_lr(opt, warm, total):
    def fn(s):
        if s < warm: return s / max(1, warm)
        return max(0., .5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    return torch.optim.lr_scheduler.LambdaLR(opt, fn)

def crit_fn(sm):
    def f(lg, t): return F.binary_cross_entropy_with_logits(
        lg, t.float() * (1 - sm) + .5 * sm, reduction="none")
    return f

def allinone_ban(ii):
    """Ban every key/value row from the query's own interaction.
    Responses are attached to their source rows and the base attention mask
    is strictly causal, so all earlier interactions remain available."""
    return ii.unsqueeze(2) == ii.unsqueeze(1)

@torch.no_grad()
def run_protocol_smoke_tests():
    """Non-trivial protocol tests: sever target labels, retain prior history."""
    dev = torch.device(DEVICE); torch.manual_seed(1701)
    q = torch.tensor([[0, 1, 2, 3, 1]], device=dev)
    s = torch.tensor([[0, 1, 2, 3, 1]], device=dev)
    r = torch.tensor([[1, 0, 0, 1, 1]], device=dev)
    a = torch.zeros_like(s); m = torch.ones_like(s, dtype=torch.float32)
    ii = torch.tensor([[0, 1, 1, 2, 2]], device=dev)
    target = (ii == 1)
    models = [("DKT", DKT(5, 5, d=32, dr=0.0)),
              ("AKT-R", AKTR(5, 5, d=32, nh=4, nl=2, dr=0.0, n_att=4)),
              ("simpleKT", SimpleKT(5, 5, d=32, nh=4, nl=2, dr=0.0, n_att=4))]
    def ai(model, rr):
        return (model.allinone_logits(q, s, rr, a, m, ii) if hasattr(model, "allinone_logits")
                else model(q, s, rr, a, m, allinone_ban(ii)))
    report = {}; max_target_delta = 0.0
    for name, model in models:
        model = model.to(dev).eval(); base = ai(model, r)
        for j in torch.where(target[0])[0].tolist():
            changed = r.clone(); changed[0, j] = 1 - changed[0, j]
            changed_pred = ai(model, changed)[target]
            max_target_delta = max(max_target_delta, float((base[target] - changed_pred).abs().max()))
            torch.testing.assert_close(base[target], changed_pred, rtol=0, atol=1e-7)
        prior = r.clone(); prior[0, 0] = 1 - prior[0, 0]
        history_delta = (base[target] - ai(model, prior)[target]).abs()
        assert torch.all(history_delta > 1e-8), f"{name}: a sibling lost preceding response history"
        if name == "DKT":
            std = model(q, s, r, a, m); first = torch.tensor([1, 3], device=dev)
            torch.testing.assert_close(std[0, first], base[0, first], rtol=0, atol=1e-7)
        report[name] = [float(x) for x in history_delta.cpu()]
    # The first canonical interaction is context only, even when rows are shuffled.
    seq = np.array([[0, 0, 0, 1, 0, 1], [0, 0, 0, 1, 0, 0],
                    [0, 0, 0, 1, 0, 2], [0, 0, 0, 1, 0, 1]], dtype=np.int32)
    sample = KTDataset({0: seq}, {0: [(0, 1, 2)]}, [0])[0]
    assert sample[-1].tolist() == [1, 0, 1, 1]
    RESULTS["protocol_tests"] = {"passed": True, "target_label_max_delta": max_target_delta,
                                 "preceding_label_delta_by_sibling": report,
                                 "canonical_first_interaction_excluded": True}
    print("Protocol smoke tests passed; preceding-label deltas by sibling:", report)

run_protocol_smoke_tests()

@torch.no_grad()
def eval_model(model, loader, ema=None, allinone=False):
    """Returns dict(row=..., fused=...). allinone=True: exact all-in-one
    protocol (faithful past context, target interaction's labels severed)."""
    model.eval()
    bm = model.module if isinstance(model, nn.DataParallel) else model
    if ema: ema.apply(bm)
    us, iis, ss, ys, ps = [], [], [], [], []
    for u, q, s, r, a, ii, m, target in loader:
        q, s, r, a, m, target = (x.to(DEVICE) for x in (q, s, r, a, m, target))
        ii_d = ii.to(DEVICE)
        if allinone and hasattr(bm, "allinone_logits"):
            logits = bm.allinone_logits(q, s, r, a, m, ii_d)
        elif allinone:
            logits = bm(q, s, r, a, m, allinone_ban(ii_d))
        else:
            logits = model(q, s, r, a, m)
        valid = (m == 1) & target
        vc = valid.cpu()
        us.append(u[vc].numpy()); iis.append(ii[vc].numpy()); ss.append(s[valid].cpu().numpy())
        ys.append(r[valid].cpu().numpy())
        ps.append(torch.sigmoid(logits[valid]).float().cpu().numpy())
    if ema: ema.restore(bm)
    if not ys: return {"row": float("nan"), "fused": float("nan"),
                       "n_rows": 0, "n_kcs": 0, "n_interactions": 0}
    u = np.concatenate(us); ii = np.concatenate(iis); s = np.concatenate(ss)
    y = np.concatenate(ys); p = np.concatenate(ps)
    raw = pd.DataFrame({"u": u, "iid": ii, "s": s, "y": y, "p": p})
    if raw.groupby(["u", "iid"])["y"].nunique().max() > 1:
        raise AssertionError("an interaction has inconsistent correctness labels")
    kc = raw.groupby(["u", "iid", "s"], sort=False).agg(y=("y", "first"), p=("p", "mean")).reset_index()
    grp = kc.groupby(["u", "iid"], sort=False).agg(y=("y", "first"), p=("p", "mean"))
    return {"row": compute_auc(y, p), "fused": compute_auc(grp["y"], grp["p"]),
            "n_rows": int(len(raw)), "n_kcs": int(len(kc)), "n_interactions": int(len(grp))}

def train_model(model_name, seqs, students, nq, ns, seed, label, window_catalog):
    """Trains with teacher forcing; selects on validation ALL-IN-ONE fused
    AUC (leak-free, uniform across conditions); reports test metrics under
    both protocols."""
    seed_everything(seed)
    trl, val, tel = make_loaders(seqs, students, nq, ns, seed, window_catalog)
    bm = build_model(model_name, nq, ns).to(DEVICE)
    model = nn.DataParallel(bm) if (DEVICE == "cuda" and torch.cuda.device_count() > 1) else bm
    n_par = sum(p.numel() for p in bm.parameters() if p.requires_grad)
    print(f"    [{label}] params={n_par:,} train_batches={len(trl)}")
    opt = torch.optim.AdamW(bm.parameters(), lr=CFG["lr"], weight_decay=CFG["wd"])
    sched = cosine_lr(opt, CFG["warmup"] * len(trl), CFG["epochs"] * len(trl))
    scaler = _scaler(CFG["use_amp"] and DEVICE == "cuda")
    ema = EMA(bm, CFG["ema_decay"])
    crit = crit_fn(CFG["label_smooth"])
    best, pat, best_state, best_shadow = -1., 0, None, None
    hist = {"epoch": [], "loss": [], "val_row": [], "val_ai_fused": []}
    for ep in range(1, CFG["epochs"] + 1):
        model.train(); ls = nb = 0
        for u, q, s, r, a, ii, m, target in trl:
            q, s, r, a, m, target = (x.to(DEVICE) for x in (q, s, r, a, m, target))
            opt.zero_grad(set_to_none=True)
            with _autocast(CFG["use_amp"] and DEVICE == "cuda"):
                logits = model(q, s, r, a, m)
                valid = (m == 1) & target
                loss = crit(logits, r)[valid].mean()
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(bm.parameters(), CFG["clip"])
            scaler.step(opt); scaler.update(); sched.step(); ema.update(bm)
            ls += loss.item(); nb += 1
        va_std = eval_model(model, val, ema, allinone=False)
        va_ai  = eval_model(model, val, ema, allinone=True)
        hist["epoch"].append(ep); hist["loss"].append(ls / max(1, nb))
        hist["val_row"].append(va_std["row"]); hist["val_ai_fused"].append(va_ai["fused"])
        crit_metric = va_ai["fused"]
        if not math.isnan(crit_metric) and crit_metric > best:
            best = crit_metric; pat = 0
            best_state = {k: v.detach().cpu().clone() for k, v in bm.state_dict().items()}
            best_shadow = {k: v.clone() for k, v in ema.s.items()}
        else:
            pat += 1
        if ep % 5 == 0 or pat == 0:
            print(f"    ep{ep:3d} loss {ls/max(1,nb):.4f} | val row {va_std['row']:.4f} "
                  f"ai-fused {va_ai['fused']:.4f} | best {best:.4f} | pat {pat}")
        if pat >= CFG["early_stop"]:
            print(f"    early stop at ep{ep}"); break
    if best_state: bm.load_state_dict(best_state); ema.s = best_shadow
    te  = eval_model(model, tel, ema, allinone=False)
    tea = eval_model(model, tel, ema, allinone=True)
    print(f"    -> TEST row={te['row']:.4f} fused={te['fused']:.4f} | "
          f"all-in-one fused={tea['fused']:.4f}")
    del model, bm
    if DEVICE == "cuda": torch.cuda.empty_cache()
    return {"test_row": te["row"], "test_fused": te["fused"],
            "test_ai_row": tea["row"], "test_ai_fused": tea["fused"],
            "test_n_rows": te["n_rows"], "test_n_kcs": te["n_kcs"],
            "test_n_interactions": te["n_interactions"],
            "best_val_ai": best, "history": hist}

Protocol smoke tests passed; preceding-label deltas by sibling: {'DKT': [0.0070976316928863525, 0.0036666393280029297], 'AKT-R': [0.12752556800842285, 0.1205371618270874], 'simpleKT': [0.4886300563812256, 0.4815745949745178]}


In [7]:
# ================================================================
# 6. SECTIONS C–F — all training runs (resumable)  (v3)
#    C: DKT / AKT-R / simpleKT × {QL, EXP} × 5 seeds   (ASSIST2009)
#    D: simpleKT × SHUF × 5 seeds                      (order-breaking)
#    E: simpleKT × LEGACY × 5 seeds                    (bug decomposition)
#    F: simpleKT × QL × 5 seeds                        (ASSIST2017; EXP
#       identical to QL there — trained only if datasets differ)
# ================================================================
A09_INDEX = {sid: i for i, sid in enumerate(A09_STUDENTS)}
A17_INDEX = {sid: i for i, sid in enumerate(A17_STUDENTS)}
A09_WINDOWS = build_window_catalog(A09_CONDS, A09_INDEX, "A09")
A17_WINDOWS = build_window_catalog(A17_CONDS, A17_INDEX, "A17")
RESULTS["stats"].setdefault("A09", dict(A09_STATS))["window_catalog"] = A09_WINDOWS["audit"]
RESULTS["stats"].setdefault("A17", dict(A17_STATS))["window_catalog"] = A17_WINDOWS["audit"]
RESULTS.setdefault("splits", {})
for ds, students in (("A09", A09_STUDENTS), ("A17", A17_STUDENTS)):
    RESULTS["splits"][ds] = {}
    for sd in SEEDS:
        tr, va, te = split_students(students, sd)
        RESULTS["splits"][ds][str(sd)] = {
            "train": [students[i] for i in tr], "validation": [students[i] for i in va],
            "test": [students[i] for i in te]}
save_results()

_PREP_CACHE = {}
def get_prepared(ds, cond):
    key = (ds, cond)
    if key not in _PREP_CACHE:
        if ds == "A09":
            _PREP_CACHE[key] = prepare_condition(A09_CONDS[cond], A09_INDEX) + (A09_STUDENTS, A09_WINDOWS)
        else:
            _PREP_CACHE[key] = prepare_condition(A17_CONDS[cond], A17_INDEX) + (A17_STUDENTS, A17_WINDOWS)
    return _PREP_CACHE[key]

RUN_PLAN = []
if RUN["C"]:
    for m in ["DKT", "AKT-R", "simpleKT"]:
        for cond in ["QL", "EXP"]:
            for sd in SEEDS: RUN_PLAN.append(("A09", m, cond, sd))
if RUN["D"]:
    for sd in SEEDS: RUN_PLAN.append(("A09", "simpleKT", "SHUF", sd))
if RUN["E"]:
    for sd in SEEDS: RUN_PLAN.append(("A09", "simpleKT", "LEGACY", sd))
if RUN["F"]:
    a17_conds = ["QL"] if A17_STATS["exp_equals_ql"] else ["QL", "EXP"]
    if A17_STATS["exp_equals_ql"]:
        print("A17: KC-expansion is a no-op (zero multi-KC interactions) — training QL only.")
    for cond in a17_conds:
        for sd in SEEDS: RUN_PLAN.append(("A17", "simpleKT", cond, sd))

todo = sum(1 for k in RUN_PLAN if "/".join(map(str, k)) not in RESULTS["runs"])
print(f"Run plan: {len(RUN_PLAN)} runs ({todo} still to do)\n")

for ds, mname, cond, sd in RUN_PLAN:
    key = f"{ds}/{mname}/{cond}/{sd}"
    if key in RESULTS["runs"]:
        r = RESULTS["runs"][key]
        print(f"[skip] {key}  (row={r['test_row']:.4f} ai-fused={r['test_ai_fused']:.4f})")
        continue
    print(f"[run ] {key}")
    t0 = time.time()
    seqs, nq, ns, students, windows = get_prepared(ds, cond)
    res = train_model(mname, seqs, students, nq, ns, sd, key, windows)
    res["minutes"] = round((time.time() - t0) / 60, 1)
    RESULTS["runs"][key] = res
    save_results()
    print(f"       done in {res['minutes']} min\n")

print("All requested runs finished.")

  [A09] shared windows: train={'windows': 4925, 'shortened_by_shared_row_budget': 257, 'max_shared_rows': 384, 'max_interactions': 200}, eval={'windows': 4599, 'shortened_by_shared_row_budget': 257, 'max_shared_rows': 384, 'max_interactions': 200}
  [A17] shared windows: train={'windows': 8644, 'shortened_by_shared_row_budget': 0, 'max_shared_rows': 200, 'max_interactions': 200}, eval={'windows': 5546, 'shortened_by_shared_row_budget': 0, 'max_shared_rows': 200, 'max_interactions': 200}
A17: KC-expansion is a no-op (zero multi-KC interactions) — training QL only.
Run plan: 45 runs (0 still to do)

[skip] A09/DKT/QL/42  (row=0.7608 ai-fused=0.7608)
[skip] A09/DKT/QL/123  (row=0.7674 ai-fused=0.7674)
[skip] A09/DKT/QL/7  (row=0.7460 ai-fused=0.7460)
[skip] A09/DKT/QL/2024  (row=0.7700 ai-fused=0.7700)
[skip] A09/DKT/QL/31  (row=0.7628 ai-fused=0.7628)
[skip] A09/DKT/EXP/42  (row=0.8326 ai-fused=0.7561)
[skip] A09/DKT/EXP/123  (row=0.8400 ai-fused=0.7642)
[skip] A09/DKT/EXP/7  (row=0.8231

In [8]:
# ================================================================
# 7. SECTION G — aggregate: paper tables + figures + zip  (v3)
#    All numbers come from RESULTS (paper_v3_results.json).
#    Paired statistics: per-seed differences share identical student
#    splits, so we report mean difference ± std and a 95% t-interval.
# ================================================================
from scipy import stats as _st

def runs(ds, model, cond, metric="test_row"):
    return [RESULTS["runs"][f"{ds}/{model}/{cond}/{sd}"][metric]
            for sd in SEEDS if f"{ds}/{model}/{cond}/{sd}" in RESULTS["runs"]]

def ms(vals):
    if not vals: return "  --  "
    return f"{np.mean(vals):.4f}±{np.std(vals, ddof=1):.4f}"

def paired(a, b):
    """a-b paired by seed: mean, std, 95% t-CI."""
    if not a or not b or len(a) != len(b): return None
    d = np.array(a) - np.array(b); n = len(d)
    m = d.mean(); s = d.std(ddof=1) if n > 1 else 0.
    h = _st.t.ppf(0.975, n - 1) * s / np.sqrt(n) if n > 1 else 0.
    return {"mean": m, "sd": s, "lo": m - h, "hi": m + h, "n": n}

def pstr(p):
    if p is None: return "--"
    return f"{p['mean']:+.4f} [{p['lo']:+.4f}, {p['hi']:+.4f}]"

if RUN["G"]:
    MODELS = ["DKT", "AKT-R", "simpleKT"]
    missing = [f"{ds}/{m}/{c}/{sd}" for ds, m, c, sd in RUN_PLAN
               if f"{ds}/{m}/{c}/{sd}" not in RESULTS["runs"]]
    if missing: raise RuntimeError(f"Aggregation requires all requested runs; missing {missing[:5]}")
    for sd in SEEDS:
        for m in MODELS:
            q = RESULTS["runs"][f"A09/{m}/QL/{sd}"]
            e = RESULTS["runs"][f"A09/{m}/EXP/{sd}"]
            assert q["test_n_interactions"] == e["test_n_interactions"], "QL/EXP target mismatch"
        e = RESULTS["runs"][f"A09/simpleKT/EXP/{sd}"]
        for cond in ("SHUF", "LEGACY"):
            z = RESULTS["runs"][f"A09/simpleKT/{cond}/{sd}"]
            assert (e["test_n_interactions"], e["test_n_kcs"]) == (z["test_n_interactions"], z["test_n_kcs"]), \
                   f"EXP/{cond} fused-target mismatch"

    print("=" * 100)
    print("TABLE 3 — ASSIST2009: QL vs clean EXP, row-level (leaky) and all-in-one (fair) evaluation")
    print(f"{'':10s} {'QL ai-fused':>16s} {'EXP row':>16s} {'EXP ai-fused':>16s} "
          f"{'gap row [95% CI]':>26s} {'gap fair [95% CI]':>26s}")
    for m in MODELS:
        ql, exr, exa = runs("A09", m, "QL"), runs("A09", m, "EXP"), runs("A09", m, "EXP", "test_ai_fused")
        qla = runs("A09", m, "QL", "test_ai_fused")
        print(f"{m:10s} {ms(qla):>16s} {ms(exr):>16s} {ms(exa):>16s} "
              f"{pstr(paired(exr, ql)):>26s} {pstr(paired(exa, qla)):>26s}")
    print("\n(QL uses its all-in-one metric — identical to row AUC there. 'gap fair' is the")
    print(" genuine effect of expanded training under leak-free selection and evaluation.)")

    print("\n" + "=" * 100)
    print("TABLE 4 — Order-breaking ablation, simpleKT (row-level eval; shared windows and targets)")
    A = runs("A09", "simpleKT", "QL"); B = runs("A09", "simpleKT", "EXP"); C = runs("A09", "simpleKT", "SHUF")
    for k, v in [("A: QL", A), ("B: EXP consecutive", B), ("C: EXP shuffled", C)]:
        print(f"  {k:22s} {ms(v)}   per-seed {[round(x,4) for x in v]}")
    print(f"  B−C (block disruption): {pstr(paired(B, C))}")
    print(f"  C−A (residual):          {pstr(paired(C, A))}")

    print("\n" + "=" * 100)
    print("TABLE 5 — Source decomposition, simpleKT (row-level eval)")
    L = runs("A09", "simpleKT", "LEGACY")
    print(f"  QL     {ms(A)}\n  EXP    {ms(B)}   (+KC-expansion    {pstr(paired(B, A))})")
    print(f"  LEGACY {ms(L)}   (+duplicate records {pstr(paired(L, B))})")

    print("\n" + "=" * 100)
    print("TABLE 6 — Evaluation-protocol contrasts (EXP-trained; sequential, not additive)")
    print(f"{'':10s} {'row':>16s} {'fused':>16s} {'ai-fused':>16s} "
          f"{'row−fused [95% CI]':>26s} {'fused−ai [95% CI]':>26s} {'ai−QL [95% CI]':>26s}")
    for m in MODELS:
        ro, fu, ai = runs("A09", m, "EXP"), runs("A09", m, "EXP", "test_fused"), runs("A09", m, "EXP", "test_ai_fused")
        qla = runs("A09", m, "QL", "test_ai_fused")
        print(f"{m:10s} {ms(ro):>16s} {ms(fu):>16s} {ms(ai):>16s} "
              f"{pstr(paired(ro, fu)):>26s} {pstr(paired(fu, ai)):>26s} {pstr(paired(ai, qla)):>26s}")
    Lf, La = runs("A09", "simpleKT", "LEGACY", "test_fused"), runs("A09", "simpleKT", "LEGACY", "test_ai_fused")
    print(f"{'sKT-LEGACY':10s} {ms(L):>16s} {ms(Lf):>16s} {ms(La):>16s} "
          f"{pstr(paired(L, Lf)):>26s} {pstr(paired(Lf, La)):>26s} "
          f"{pstr(paired(La, runs('A09','simpleKT','QL','test_ai_fused'))):>26s}")

    print("\n" + "=" * 100)
    print("TABLE 7 — Cross-dataset reference (simpleKT, all-in-one eval)")
    for ds, lbl in [("A09", "ASSIST2009"), ("A17", "ASSIST2017")]:
        qla = runs(ds, "simpleKT", "QL", "test_ai_fused")
        note = " (KC-expansion is a no-op: datasets identical)" if RESULTS["stats"][ds].get("exp_equals_ql") else ""
        print(f"  {lbl:12s} multi-KC {RESULTS['stats'][ds]['multi_kc_pct']:5.2f}%   QL {ms(qla)}{note}")

    # ── Figure 3: protocol comparison ───────────────────────────
    fig, ax = plt.subplots(figsize=(11, 6))
    x = np.arange(len(MODELS)); w = 0.26
    for off, cond, metric, color, lab in [
            (-w, "EXP", "test_row",      "#F44336", "EXP-trained, row eval (leaky)"),
            (0., "EXP", "test_ai_fused", "#FF9800", "EXP-trained, all-in-one eval (fair)"),
            (w,  "QL",  "test_ai_fused", "#1976D2", "QL-trained (fair)")]:
        mu = [np.mean(runs("A09", m, cond, metric) or [np.nan]) for m in MODELS]
        er = [np.std(runs("A09", m, cond, metric) or [np.nan], ddof=1) for m in MODELS]
        ax.bar(x + off, mu, w, yerr=er, capsize=4, color=color, alpha=0.9, label=lab)
        for xi, m_ in zip(x + off, mu):
            if not math.isnan(m_): ax.text(xi, m_ + 0.004, f"{m_:.3f}", ha="center", fontsize=9, fontweight="bold")
    ax.set_xticks(x); ax.set_xticklabels(MODELS, fontsize=12)
    ax.set_ylabel("Test AUC"); ax.set_ylim(0.6, 1.0)
    ax.set_title("ASSIST2009: training protocol × evaluation protocol (mean±std, 5 seeds)")
    ax.legend(fontsize=10); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig3_multi_model.png", dpi=150, bbox_inches="tight"); plt.close()

    # ── Figure 4: order-breaking ────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    names = ["A: Question-level", "B: KC-expanded\nconsecutive", "C: KC-expanded\nshuffled"]
    mu = [np.mean(v or [np.nan]) for v in (A, B, C)]
    er = [np.std(v or [np.nan], ddof=1) for v in (A, B, C)]
    axes[0].bar(names, mu, 0.55, yerr=er, capsize=5, color=["#1976D2", "#F44336", "#FF9800"])
    for xi, m_ in enumerate(mu):
        axes[0].text(xi, m_ + 0.005, f"{m_:.4f}", ha="center", fontsize=11, fontweight="bold")
    axes[0].set_ylabel("Test AUC, row eval (simpleKT)"); axes[0].set_ylim(0.65, 0.95)
    axes[0].set_title("Order-breaking ablation (mean±std, 5 seeds)\nB and C: same rows — only ordering differs")
    for c, lab, col, sty in [("QL", "A: Q-level", "#1976D2", "-"),
                             ("EXP", "B: KC-exp consec.", "#F44336", "-"),
                             ("SHUF", "C: KC-exp shuffled", "#FF9800", "--")]:
        k = f"A09/simpleKT/{c}/{MAIN_SEED}"
        if k in RESULTS["runs"]:
            h = RESULTS["runs"][k]["history"]
            axes[1].plot(h["epoch"], h["val_row"], sty, color=col, label=lab)
    axes[1].set_xlabel("epoch"); axes[1].set_ylabel("validation AUC (row)")
    axes[1].set_title(f"Validation curves (seed {MAIN_SEED})"); axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig4_order_breaking.png", dpi=150, bbox_inches="tight"); plt.close()

    # ── Figure 5: source decomposition ──────────────────────────
    fig, ax = plt.subplots(figsize=(8, 5))
    names = ["QL\n(clean)", "EXP\n(KC-expansion)", "LEGACY\n(KC-exp + bug)"]
    mu = [np.mean(v or [np.nan]) for v in (A, B, L)]
    er = [np.std(v or [np.nan], ddof=1) for v in (A, B, L)]
    ax.bar(names, mu, 0.5, yerr=er, capsize=5, color=["#1976D2", "#FF9800", "#B71C1C"])
    for xi, m_ in enumerate(mu):
        ax.text(xi, m_ + 0.004, f"{m_:.4f}", ha="center", fontsize=11, fontweight="bold")
    ax.set_ylabel("Test AUC, row eval (simpleKT)"); ax.set_ylim(0.65, 0.95)
    ax.set_title("Row-level inflation by duplicate source (mean±std, 5 seeds)")
    plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig5_decomposition.png", dpi=150, bbox_inches="tight"); plt.close()

    # ── Figure 6: evaluation contrasts ──────────────────────────
    fig, ax = plt.subplots(figsize=(11, 6))
    x = np.arange(len(MODELS)); w = 0.2
    bars = [("QL-trained (fair)", [runs("A09", m, "QL", "test_ai_fused") for m in MODELS], "#1976D2"),
            ("EXP all-in-one (fair)", [runs("A09", m, "EXP", "test_ai_fused") for m in MODELS], "#4CAF50"),
            ("EXP fused (partial leak)", [runs("A09", m, "EXP", "test_fused") for m in MODELS], "#FF9800"),
            ("EXP row (leaky)", [runs("A09", m, "EXP", "test_row") for m in MODELS], "#F44336")]
    for i, (lab, vals, col) in enumerate(bars):
        off = (i - 1.5) * w
        mu = [np.mean(v or [np.nan]) for v in vals]; er = [np.std(v or [np.nan], ddof=1) for v in vals]
        ax.bar(x + off, mu, w, yerr=er, capsize=3, color=col, alpha=0.9, label=lab)
        for xi, m_ in zip(x + off, mu):
            if not math.isnan(m_): ax.text(xi, m_ + 0.004, f"{m_:.3f}", ha="center", fontsize=8, fontweight="bold")
    ax.set_xticks(x); ax.set_xticklabels(MODELS, fontsize=12)
    ax.set_ylabel("Test AUC"); ax.set_ylim(0.65, 0.95)
    ax.set_title("Evaluation-protocol contrasts on ASSIST2009 (mean±std, 5 seeds)\n"
                 "sequential contrasts — see text; AUC differences are not additive channels")
    ax.legend(fontsize=10); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig6_eval_contrasts.png", dpi=150, bbox_inches="tight"); plt.close()

    save_results()
    print(f"\nFigures + results saved in {OUT_DIR}/")
    import shutil
    zip_path = shutil.make_archive("paper_v3_results", "zip", OUT_DIR)
    print(f"Download: {zip_path}")

TABLE 3 — ASSIST2009: QL vs clean EXP, row-level (leaky) and all-in-one (fair) evaluation
                QL ai-fused          EXP row     EXP ai-fused           gap row [95% CI]          gap fair [95% CI]
DKT           0.7614±0.0093    0.8355±0.0087    0.7570±0.0092 +0.0741 [+0.0710, +0.0772] -0.0044 [-0.0058, -0.0030]
AKT-R         0.7442±0.0100    0.8182±0.0105    0.7337±0.0119 +0.0740 [+0.0693, +0.0787] -0.0105 [-0.0165, -0.0044]
simpleKT      0.7678±0.0079    0.7921±0.0085    0.7633±0.0085 +0.0243 [+0.0180, +0.0307] -0.0044 [-0.0086, -0.0003]

(QL uses its all-in-one metric — identical to row AUC there. 'gap fair' is the
 genuine effect of expanded training under leak-free selection and evaluation.)

TABLE 4 — Order-breaking ablation, simpleKT (row-level eval; shared windows and targets)
  A: QL                  0.7678±0.0079   per-seed [0.7717, 0.7754, 0.7568, 0.7624, 0.7728]
  B: EXP consecutive     0.7921±0.0085   per-seed [0.7988, 0.7923, 0.7782, 0.7921, 0.7993]
  C: EXP shuff

## After the run

Download **`paper_v3_results.zip`**: `paper_v3_results.json` (all per-seed metrics under both
evaluation protocols, histories, shared-window audit, exact split IDs, data hashes, protocol
tests, stats, and shortcut results) plus figures `fig1`–`fig6`.

Bring it back and the paper is regenerated from this single JSON (tables, prose numbers, paired
confidence intervals, and figures all from the same source).

**If a session times out:** copy a partial result produced by this v3.2 notebook into
`/kaggle/working/paper_v3_out/` and re-run — finished runs are skipped automatically.

In [9]:
# ================================================================
# TEST 4 - ASSIST2009 INTERACTION-AWARE, LEAK-FREE EXP TRAINING
# Paste this cell AFTER the completed kc-experiment-v3 notebook cells.
#
# Scientific contrast:
#   conventional EXP training -> teacher-forced sibling-label paths
#   EXP-AIT training          -> those paths are removed during training
#
# Everything else is held fixed: clean EXP data, student splits, shared
# windows, architectures, seeds, optimizer, hyperparameters, EMA,
# checkpoint objective, and exact all-in-one evaluation.
#
# The loss remains row-weighted deliberately. EXP-AIT versus EXP therefore
# changes the forward information paths without changing target weighting.
# ================================================================
import os, time, math, shutil
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from scipy import stats as _ia_stats

IA_MODELS = ["DKT", "AKT-R", "simpleKT"]
IA_SEEDS = list(SEEDS)
IA_CONDITION = "EXP-AIT"
IA_FORCE_RERUN = False

_required = [
    "A09_CONDS", "A09_INDEX", "A09_WINDOWS", "A09_STUDENTS", "RESULTS",
    "prepare_condition", "make_loaders", "build_model", "allinone_ban",
    "eval_model", "EMA", "crit_fn", "cosine_lr", "save_results",
]
_missing = [name for name in _required if name not in globals()]
if _missing:
    raise RuntimeError(
        "Run kc-experiment-v3 through its aggregation cell first. "
        f"Missing objects: {_missing}"
    )


class _InteractionAwareForward(nn.Module):
    """Expose the notebook's exact all-in-one prediction as forward()."""
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model

    def forward(self, q, s, r, a, mask, interaction_id):
        # DKT: all siblings use the state at the previous interaction boundary.
        if hasattr(self.base_model, "allinone_logits"):
            return self.base_model.allinone_logits(
                q, s, r, a, mask, interaction_id
            )
        # Transformers: ban every key/value row from the query's interaction.
        return self.base_model(
            q, s, r, a, mask, allinone_ban(interaction_id)
        )


def _train_model_interaction_aware(
    model_name, seqs, students, nq, ns, seed, label, window_catalog
):
    """Train clean EXP from scratch under exact all-in-one restrictions."""
    seed_everything(seed)
    train_loader, validation_loader, test_loader = make_loaders(
        seqs, students, nq, ns, seed, window_catalog
    )

    base_model = build_model(model_name, nq, ns).to(DEVICE)
    ia_forward = _InteractionAwareForward(base_model).to(DEVICE)
    training_forward = (
        nn.DataParallel(ia_forward)
        if DEVICE == "cuda" and torch.cuda.device_count() > 1
        else ia_forward
    )

    n_parameters = sum(
        parameter.numel()
        for parameter in base_model.parameters()
        if parameter.requires_grad
    )
    print(
        f"    [{label}] params={n_parameters:,} "
        f"train_batches={len(train_loader)}"
    )

    optimizer = torch.optim.AdamW(
        base_model.parameters(), lr=CFG["lr"], weight_decay=CFG["wd"]
    )
    scheduler = cosine_lr(
        optimizer,
        CFG["warmup"] * len(train_loader),
        CFG["epochs"] * len(train_loader),
    )
    scaler = _scaler(CFG["use_amp"] and DEVICE == "cuda")
    ema = EMA(base_model, CFG["ema_decay"])
    criterion = crit_fn(CFG["label_smooth"])

    best = -1.0
    patience = 0
    best_state = None
    best_shadow = None
    history = {"epoch": [], "loss": [], "val_ai_fused": []}

    for epoch in range(1, CFG["epochs"] + 1):
        training_forward.train()
        loss_sum = 0.0
        n_batches = 0

        for u, q, s, r, a, interaction_id, mask, target in train_loader:
            q, s, r, a, interaction_id, mask, target = (
                tensor.to(DEVICE)
                for tensor in (q, s, r, a, interaction_id, mask, target)
            )
            optimizer.zero_grad(set_to_none=True)

            with _autocast(CFG["use_amp"] and DEVICE == "cuda"):
                logits = training_forward(q, s, r, a, mask, interaction_id)
                valid = (mask == 1) & target
                loss = criterion(logits, r)[valid].mean()

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(base_model.parameters(), CFG["clip"])
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            ema.update(base_model)

            loss_sum += float(loss.item())
            n_batches += 1

        # Same leak-free checkpoint objective used in the main notebook.
        validation_ai = eval_model(
            base_model, validation_loader, ema, allinone=True
        )
        validation_score = validation_ai["fused"]
        mean_loss = loss_sum / max(1, n_batches)

        history["epoch"].append(epoch)
        history["loss"].append(mean_loss)
        history["val_ai_fused"].append(validation_score)

        if not math.isnan(validation_score) and validation_score > best:
            best = validation_score
            patience = 0
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in base_model.state_dict().items()
            }
            best_shadow = {
                key: value.clone() for key, value in ema.s.items()
            }
        else:
            patience += 1

        if epoch % 5 == 0 or patience == 0:
            print(
                f"    ep{epoch:3d} loss {mean_loss:.4f} | "
                f"val ai-fused {validation_score:.4f} | "
                f"best {best:.4f} | pat {patience}"
            )
        if patience >= CFG["early_stop"]:
            print(f"    early stop at ep{epoch}")
            break

    if best_state is None:
        raise RuntimeError(f"{label}: no valid checkpoint was produced")

    base_model.load_state_dict(best_state)
    ema.s = best_shadow

    # Standard scores are diagnostic only. The primary comparisons below use
    # exact all-in-one interaction-fused AUC.
    test_standard = eval_model(
        base_model, test_loader, ema, allinone=False
    )
    test_ai = eval_model(base_model, test_loader, ema, allinone=True)
    print(
        f"    -> TEST row={test_standard['row']:.4f} "
        f"fused={test_standard['fused']:.4f} | "
        f"all-in-one fused={test_ai['fused']:.4f}"
    )

    result = {
        "training_protocol": (
            "exact all-in-one forward; row-weighted smoothed BCE"
        ),
        "test_row": test_standard["row"],
        "test_fused": test_standard["fused"],
        "test_ai_row": test_ai["row"],
        "test_ai_fused": test_ai["fused"],
        "test_n_rows": test_standard["n_rows"],
        "test_n_kcs": test_standard["n_kcs"],
        "test_n_interactions": test_standard["n_interactions"],
        "best_val_ai": best,
        "history": history,
    }

    del training_forward, ia_forward, base_model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return result


def _paired_summary(a, b):
    """Paired a-b t interval over identical seeds and student splits."""
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    if len(a) != len(b) or len(a) == 0:
        raise ValueError("Paired samples must be non-empty and equal in length")
    differences = a - b
    n = len(differences)
    mean = float(differences.mean())
    sd = float(differences.std(ddof=1)) if n > 1 else 0.0
    half_width = (
        float(_ia_stats.t.ppf(0.975, n - 1) * sd / np.sqrt(n))
        if n > 1
        else 0.0
    )
    return {
        "mean": mean,
        "sd": sd,
        "lo": mean - half_width,
        "hi": mean + half_width,
        "n": n,
        "per_seed": differences.tolist(),
    }


# Reuse the exact clean EXP representation and shared interaction windows.
_ia_seqs, _ia_nq, _ia_ns = prepare_condition(
    A09_CONDS["EXP"], A09_INDEX
)

RESULTS.setdefault("interaction_aware_training", {})["design"] = {
    "condition": IA_CONDITION,
    "dataset": "A09",
    "training_representation": "clean EXP",
    "information_protocol": "exact all-in-one during training",
    "loss_weighting": "row-weighted, matching conventional EXP",
    "checkpoint_objective": "validation all-in-one fused AUC",
    "primary_test_metric": "test all-in-one fused AUC",
    "paired_seeds": list(IA_SEEDS),
    "only_intended_change_vs_EXP": (
        "training-time same-interaction label paths removed"
    ),
}
save_results()

_ia_plan = [
    (model_name, seed)
    for model_name in IA_MODELS
    for seed in IA_SEEDS
]
_ia_todo = sum(
    IA_FORCE_RERUN
    or f"A09/{model_name}/{IA_CONDITION}/{seed}" not in RESULTS["runs"]
    for model_name, seed in _ia_plan
)
print(
    f"\nASSIST2009 Test 4 plan: {len(_ia_plan)} runs "
    f"({_ia_todo} still to do; saved after each run)\n"
)

for _ia_model_name, _ia_seed in _ia_plan:
    _ia_key = f"A09/{_ia_model_name}/{IA_CONDITION}/{_ia_seed}"
    if _ia_key in RESULTS["runs"] and not IA_FORCE_RERUN:
        _ia_result = RESULTS["runs"][_ia_key]
        print(
            f"[skip] {_ia_key} "
            f"(ai-fused={_ia_result['test_ai_fused']:.4f})"
        )
        continue

    print(f"[run ] {_ia_key}")
    _ia_start = time.time()
    _ia_result = _train_model_interaction_aware(
        _ia_model_name,
        _ia_seqs,
        A09_STUDENTS,
        _ia_nq,
        _ia_ns,
        _ia_seed,
        _ia_key,
        A09_WINDOWS,
    )
    _ia_result["minutes"] = round(
        (time.time() - _ia_start) / 60, 1
    )

    _exp_key = f"A09/{_ia_model_name}/EXP/{_ia_seed}"
    if _exp_key not in RESULTS["runs"]:
        raise RuntimeError(f"Missing conventional comparison run: {_exp_key}")
    _exp_result = RESULTS["runs"][_exp_key]
    assert (
        _ia_result["test_n_rows"],
        _ia_result["test_n_kcs"],
        _ia_result["test_n_interactions"],
    ) == (
        _exp_result["test_n_rows"],
        _exp_result["test_n_kcs"],
        _exp_result["test_n_interactions"],
    ), f"{_ia_key}: target counts do not match conventional EXP"

    RESULTS["runs"][_ia_key] = _ia_result
    save_results()
    print(f"       done in {_ia_result['minutes']} min\n")


def _values(model_name, condition, metric="test_ai_fused"):
    return [
        RESULTS["runs"][f"A09/{model_name}/{condition}/{seed}"][metric]
        for seed in IA_SEEDS
    ]


_ia_summary = {}
print("=" * 108)
print(
    "ASSIST2009 TEST 4 - leak-free EXP training; "
    "fair metric = all-in-one fused AUC"
)
print(
    f"{'model':10s} {'QL mean+/-sd':>16s} {'EXP mean+/-sd':>16s} "
    f"{'EXP-AIT mean+/-sd':>19s} {'AIT-EXP [95% CI]':>23s} "
    f"{'AIT-QL [95% CI]':>23s}"
)

for _ia_model_name in IA_MODELS:
    _ql = _values(_ia_model_name, "QL")
    _exp = _values(_ia_model_name, "EXP")
    _ait = _values(_ia_model_name, IA_CONDITION)
    _ait_minus_exp = _paired_summary(_ait, _exp)
    _ait_minus_ql = _paired_summary(_ait, _ql)
    _ia_summary[_ia_model_name] = {
        "ql_ai_fused": _ql,
        "exp_ai_fused": _exp,
        "exp_ait_ai_fused": _ait,
        "exp_ait_minus_exp": _ait_minus_exp,
        "exp_ait_minus_ql": _ait_minus_ql,
    }

    def _msd(values):
        return f"{np.mean(values):.4f}+/-{np.std(values, ddof=1):.4f}"

    def _pci(result):
        return (
            f"{result['mean']:+.4f} "
            f"[{result['lo']:+.4f}, {result['hi']:+.4f}]"
        )

    print(
        f"{_ia_model_name:10s} {_msd(_ql):>16s} {_msd(_exp):>16s} "
        f"{_msd(_ait):>19s} {_pci(_ait_minus_exp):>23s} "
        f"{_pci(_ait_minus_ql):>23s}"
    )

RESULTS["interaction_aware_training"]["summary"] = _ia_summary
RESULTS["interaction_aware_training"]["tests"] = {
    "completed_runs": len(_ia_plan),
    "target_counts_match_conventional_exp": True,
}
save_results()


# Manuscript-ready figure from the saved per-seed results.
fig, ax = plt.subplots(figsize=(11, 6))
_x = np.arange(len(IA_MODELS))
_width = 0.24
_series = [
    (-_width, "QL", "#1976D2", "QL-trained"),
    (0.0, "EXP", "#F57C00", "EXP-trained (conventional)"),
    (_width, IA_CONDITION, "#2E7D32", "EXP-trained (leak-free)"),
]
for _offset, _condition, _color, _label in _series:
    _all_values = [_values(model, _condition) for model in IA_MODELS]
    _means = [float(np.mean(values)) for values in _all_values]
    _errors = [float(np.std(values, ddof=1)) for values in _all_values]
    _bars = ax.bar(
        _x + _offset,
        _means,
        _width,
        yerr=_errors,
        capsize=4,
        color=_color,
        alpha=0.9,
        label=_label,
    )
    for _bar, _value in zip(_bars, _means):
        ax.text(
            _bar.get_x() + _bar.get_width() / 2,
            _value + 0.004,
            f"{_value:.3f}",
            ha="center",
            fontsize=8,
            fontweight="bold",
        )

ax.set_xticks(_x)
ax.set_xticklabels(IA_MODELS)
ax.set_ylabel("test AUC (all-in-one, interaction-fused)")
ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
ax.set_ylim(0.60, 0.95)
ax.set_title(
    "ASSIST2009: leak-free EXP training\n"
    "mean +/- sample SD over five paired student splits"
)
ax.legend(fontsize=9)
plt.tight_layout()
_ia_figure_path = os.path.join(
    OUT_DIR, "fig7_interaction_aware_training.png"
)
plt.savefig(_ia_figure_path, dpi=150, bbox_inches="tight")
plt.close()

# Refresh the notebook's standard downloadable archive.
_ia_archive = shutil.make_archive("paper_v3_results", "zip", OUT_DIR)
print(f"\nSaved: {_ia_figure_path}")
print(f"Updated JSON: {RESULTS_PATH}")
print(f"Download: {_ia_archive}")



ASSIST2009 Test 4 plan: 15 runs (15 still to do; saved after each run)

[run ] A09/DKT/EXP-AIT/42
    [A09/DKT/EXP-AIT/42] params=146,812 train_batches=31
    ep  1 loss 0.6899 | val ai-fused 0.5143 | best 0.5143 | pat 0
    ep  2 loss 0.6494 | val ai-fused 0.5385 | best 0.5385 | pat 0
    ep  3 loss 0.6025 | val ai-fused 0.5739 | best 0.5739 | pat 0
    ep  4 loss 0.5812 | val ai-fused 0.6234 | best 0.6234 | pat 0
    ep  5 loss 0.5729 | val ai-fused 0.6690 | best 0.6690 | pat 0
    ep  6 loss 0.5679 | val ai-fused 0.6951 | best 0.6951 | pat 0
    ep  7 loss 0.5645 | val ai-fused 0.7087 | best 0.7087 | pat 0
    ep  8 loss 0.5622 | val ai-fused 0.7169 | best 0.7169 | pat 0
    ep  9 loss 0.5598 | val ai-fused 0.7226 | best 0.7226 | pat 0
    ep 10 loss 0.5577 | val ai-fused 0.7271 | best 0.7271 | pat 0
    ep 11 loss 0.5559 | val ai-fused 0.7307 | best 0.7307 | pat 0
    ep 12 loss 0.5550 | val ai-fused 0.7339 | best 0.7339 | pat 0
    ep 13 loss 0.5532 | val ai-fused 0.7365 | best 0